# POLARIS RIO Calculator

This notebook contains a Python function to compute the **Risk Index Outcome (RIO)** for a vessel
based on POLARIS guidance (IMO MSC.1/Circ.1519). It includes:

- the `polar_rio` function (with an approximate thickness→ice-type mapping),
- usage examples, and
- a small demo table of example inputs and outputs.

The code in this notebook was generated by an assistant and includes docstrings and example calls.

— Generated: 2025-12-19 17:47:23 UTC

In [ ]:
# POLARIS RIO calculator (approximate thickness->ice_type mapping included)
from typing import Tuple, Optional

ICE_TYPE_KEYS = [
    "ice_free", "new_ice", "grey_ice", "grey_white_ice",
    "thin_first_year_1st_stage", "thin_first_year_2nd_stage", "thin_first_year",
    "medium_first_year_lt_1m", "medium_first_year", "thick_first_year",
    "second_year_ice", "light_multi_year_lt2p5", "heavy_multi_year"
]

RIV_TABLE = {
    "PC1":  [3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 1, 1, 1],
    "PC2":  [3, 3, 3, 3, 2, 2, 2, 2, 2, 1, 1, 1, 0],
    "PC3":  [3, 3, 3, 3, 2, 2, 2, 2, 2, 1, 0, -1, -1],
    "PC4":  [3, 3, 3, 3, 2, 2, 2, 2, 1, 0, -1, -2, -2],
    "PC5":  [3, 3, 3, 3, 2, 2, 1, 1, 0, -1, -2, -2, -2],
    "PC6":  [3, 2, 2, 2, 2, 1, 1, 0, -1, -2, -3, -3, -3],
    "PC7":  [3, 2, 2, 2, 1, 1, 0, -1, -2, -3, -3, -3, -3],
    "IA Super": [3, 2, 2, 2, 2, 1, 0, -1, -2, -3, -4, -4, -4],
    "IA":   [3, 2, 2, 2, 1, 0, -1, -2, -3, -4, -5, -5, -5],
    "IB":   [3, 2, 2, 1, 0, -1, -2, -3, -4, -5, -6, -6, -6],
    "IC":   [3, 2, 1, 0, -1, -2, -3, -4, -5, -6, -7, -8, -8],
    "NIS": [3, 1, 0, -1, -2, -3, -4, -5, -6, -7, -8, -8, -8],  # NIS = 'Not Ice Strengthened'
}

def _approx_thickness_to_ice_type(thickness_m: float) -> str:
    if thickness_m is None:
        return "ice_free"
    if thickness_m <= 0.0:
        return "ice_free"
    if thickness_m <= 0.05:
        return "new_ice"
    if thickness_m <= 0.30:
        return "grey_ice"
    if thickness_m <= 0.50:
        return "grey_white_ice"
    if thickness_m <= 0.70:
        return "thin_first_year_1st_stage"
    if thickness_m <= 1.00:
        return "thin_first_year_2nd_stage"
    if thickness_m <= 1.25:
        return "thin_first_year"
    if thickness_m <= 1.75:
        return "medium_first_year_lt_1m"
    if thickness_m <= 2.00:
        return "medium_first_year"
    if thickness_m <= 2.25:
        return "thick_first_year"
    if thickness_m <= 2.75:
        return "second_year_ice"
    if thickness_m <= 2.5:
        return "light_multi_year_lt2p5"
    return "heavy_multi_year"

def polar_rio(
    vessel_ice_class: str,
    ice_thickness_m: Optional[float] = None,
    ice_type: Optional[str] = None,
    concentration_tenths: int = 10
) -> Tuple[float, str, str]:
    """Compute POLARIS RIO for a single ice type covering the observed regime."""
    cls = vessel_ice_class.strip()
    if cls not in RIV_TABLE:
        raise ValueError(f"Unknown ice class '{vessel_ice_class}'. Valid: {list(RIV_TABLE.keys())}")
    if ice_type is None:
        if ice_thickness_m is None:
            raise ValueError("Either ice_type or ice_thickness_m must be provided.")
        ice_type_used = _approx_thickness_to_ice_type(float(ice_thickness_m))
    else:
        ice_type_used = ice_type.strip()
        if ice_type_used not in ICE_TYPE_KEYS:
            raise ValueError(f"Unknown ice_type '{ice_type}'. Valid keys: {ICE_TYPE_KEYS}")

    idx = ICE_TYPE_KEYS.index(ice_type_used)
    riv = RIV_TABLE[cls][idx]
    c = int(round(concentration_tenths))
    if c < 0 or c > 10:
        raise ValueError("concentration_tenths must be between 0 and 10 (tenths).")

        # Altered version of originally proposed code to account for RIV associated w/ open water 
        # rio = concentration_tenths * riv  # original code
        rio = concentration_tenths * riv + (10 - concentration_tenths) * 3

    is_pc1_7 = cls.upper().startswith("PC") and cls[2:].isdigit() and 1 <= int(cls[2:]) <= 7
    if is_pc1_7:
        if rio >= 0:
            op = "Normal operation"
        elif rio >= -10:
            op = "Elevated operational risk"
        else:
            op = "Operation subject to special consideration"
    else:
        if rio >= 0:
            op = "Normal operation"
        else:
            op = "Operation subject to special consideration"

    return rio, ice_type_used, op


In [ ]:
# Examples / Demo
examples = [
    ("PC3", 0.8, None, 10),   # PC3, 0.8 m thickness, 100% conc.
    ("IA", None, "grey_ice", 6),  # IA, 6 tenths grey ice
    ("PC6", 2.8, None, 10),   # PC6, 2.8 m thickness (multi-year), 100%
    ("Not Ice Strengthened", 0.2, None, 10),
]

results = []
for cls, thickness, itype, conc in examples:
    rio, used, level = polar_rio(cls, ice_thickness_m=thickness, ice_type=itype, concentration_tenths=conc)
    results.append((cls, thickness, itype, conc, rio, used, level))

from pandas import DataFrame
df = DataFrame(results, columns=["ice_class", "thickness_m", "ice_type_given", "conc_tenths", "RIO", "ice_type_used", "operation_level"])
df
